# Approach 2 — Statistics Embedded in Prompt (No Pre-Screening)

**Thesis Section: Chapter 4.2**

Approach 2 embeds signal statistics directly into the question text and
sends the middle 512 points to ChatTS-14B — no pre-screening, no chunk selection.

The model receives:
- Global mean, std, min, max
- 24hr rolling mean range (drift indicator)
- Minimum 10-pt rolling std (frozen indicator)
- 3-sigma thresholds (spike indicators)

**Difference from Approach 3**: No pre-screener, always middle chunk, no template switching.

In [ ]:
import sys
sys.path.insert(0, '../..')

from dotenv import load_dotenv
load_dotenv('../../.env')

import torch
from src.data.timeseer_client import fetch_series_api, list_series_api, AREAS
from src.data.ground_truth import GROUND_TRUTH
from src.models.chatts_loader import load_model
from src.prompts.builder import build_smart_question
from src.inference.chatts_infer import ask_chatts_chunk
from src.parsing.extract import extract_category
from src.evaluation.metrics import compute_metrics
from src.evaluation.report import print_summary_table, save_results

print('Imports OK')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
model, tokenizer, processor = load_model('ChatTS-14B')
ACTIVE_MODEL = 'ChatTS-14B'

In [ ]:
AREA    = 'Reactor 1'
tags    = list_series_api(AREA)
results = []

for tag in tags:
    print(f'\n── {tag} ──')
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None:
        print('  FAILED')
        continue

    # Approach 2: stats embedded, no pre-screening
    chunk, question = build_smart_question(vals, idx, tag)
    print(f'  Stats: mean={vals.mean():.3f} std={vals.std():.3f} '
          f'min={vals.min():.3f} max={vals.max():.3f}')
    print(f'  Chunk: {len(chunk)} points (middle of dataset)')

    torch.cuda.empty_cache()
    answer = ask_chatts_chunk([chunk], question,
                              model=model, tokenizer=tokenizer, processor=processor)
    cat_code, cat_label = extract_category(answer)

    gt = GROUND_TRUTH.get(tag, '?')
    results.append({'Tag': tag, 'gt': gt, 'Category': cat_code,
                    'Label': cat_label, 'Answer': answer[:400]})
    print(f'  ChatTS → {cat_code}) {cat_label}  (GT: {gt})')

In [ ]:
print_summary_table(results, title=f'APPROACH 2 — {AREA}', model_name=ACTIVE_MODEL)

preds = [r['Category'] for r in results if r['gt'] != '?']
gts   = [r['gt']       for r in results if r['gt'] != '?']
m = compute_metrics(preds, gts)
print(f'\nAccuracy: {m["accuracy"]:.3f}  F1: {m["f1"]:.3f}')

save_results(results, f'../../data/chatts14b_approach2_{AREA.replace(" ","_")}.txt',
             header=f'ChatTS Approach 2 — Stats Embedded | {AREA} | {ACTIVE_MODEL}')